# Lab 2: In Class Practice

## Tensile Testing of Metals

This is a guided **teaching notebook** with a mix of written response and coding for use in class. 

This notebook is designed to be used during class. The full notebook is found here: [Lab 2: Tensile Testing Metals](https://chems.gitbook.io/Notebooks/mse/250/lab_2_tensile_testing_metals)


## How to use this notebook

This notebook contains two main cell types:

- **Markdown cells** contain instructions, formatted text, and your written answers. This is a markdown cell.
- **Code cells** contain Python. Run a selected cell with **Shift+Enter**.

Python remembers variables created by earlier cells. For that reason, you need to run the code cells from top to bottom. If a later cell reports that a name is undefined, return to the top and run all the cells before continuing.

Useful habits:

1. Read the text above a code cell.
1. Run the cell without changing it unless the instructions explicitly ask you to.
1. Inspect the table, number, or graph that appears.
1. Check whether the result is physically reasonable.

If you run into a coding error, you may use AI (i.e., a large language model) to troubleshoot it. Provide the code and the error mesage to AI to help you work through the problem. Note, however, that AI may not be used to analyze the data or answer the written questions for you.


## 1. Load the tensile test data with Python

This section gives you the code to import your data into the notebook. The data must be saved as a `.csv` file. This is the default format from the Instron tensile test machine.

Place the tensile test `.csv` files in the same folder as this notebook. The files need to be visible to the notebook so they can be loaded in.

Run the code below to import the Python libraries you'll use in this notebook.


In [2]:
# Import necessary libraries
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

print("Modules imported successfully.")


Modules imported successfully.


### 2.1 Check the location of your data

Run the code below. Every ouput line should report a found data file. You need to change the name of the `.csv` file in the code to match your file name, or vice versa.

If your file is listed as not present:
- ensure your data file is in the same folder as this notebook.
- compare the name of your data files with the name given in the code below. They need to match exactly.


*Make note of the sample names*: `Al 2024`, `Al 6061`, `Brass`, `Steel`

In [6]:
# Set up the data directory and file path
DATA_DIR = Path("./")

FILES = {
    "My Material": DATA_DIR / "AL2024_1.csv",
}

# Check that the file exists
for material, path in FILES.items():
    status = "found" if path.exists() else "is not present"
    print(f"{material}: data file '{path}' {status}")


My Material: data file 'AL2024_1.csv' found


### 2.2 Inspect your .csv files

**`ENGINEERING CHECK`**:

Open and inspect the `.csv` data files provided by the Instron tensile test machine. You can open the data in Jupyter Hub by double-clicking on the .csv file.

What are the units of each measurement?

- **Specimen Width unit**:  *your response here*
- **Specimen Thickness unit**:  *your response here*
- **Specimen Gauge Length unit**:  *your response here*


Fill in the next cell manually with information from the `.csv` files, then run it.


In [ ]:
# STUDENT: manually input the specimen dimensions for all materials.
SPECIMEN_DIMENSIONS = {
    "My Material": [
        {"width": 0.0, "thickness": 0.0, "gauge_length": 0.0, "label": "1"},        # STUDENT: replace 0.0 with the measured values from .csv
    ],
}

invalid_values = [
    value
    for material_dimensions in SPECIMEN_DIMENSIONS.values()
    for specimen in material_dimensions
    for value in specimen.values()
    if isinstance(value, (int, float)) and value <= 0
]

if invalid_values:
    print("There was a problem. Please check your inputs; all values must be strictly positive (>0).")
else:
    print("User input accepted.")


User input accepted.


### 2.3 Load your data into this Notebook

Run the code cell below. You are **not expected to write or memorize this code**. Read the comments to understand each section and run it without alterations.

In [9]:
# Define a function to read measurement runs from one Instron CSV file.
def read_instron_runs(path):
    with path.open(newline="", encoding="utf-8-sig") as handle:
        content = list(csv.reader(handle))

    begin = [
        i for i, line in enumerate(content)
        if len(line) >= 2
        and line[0].strip().isdigit()
        and line[1].strip().lower() == "time"
    ]

    if not begin:
        raise ValueError(f"{path.name}: no measurement blocks were found")

    end = begin[1:] + [len(content)]
    runs = []

    for start, stop in zip(begin, end):
        rows = []

        for row in content[start + 2:stop]:
            if len(row) < 4:
                continue

            try:
                rows.append([float(value) for value in row[1:4]])
            except ValueError:
                continue

        if not rows:
            raise ValueError(f"{path.name}: measurement block contains no numeric data")

        runs.append(pd.DataFrame(
            rows,
            columns=["Time/s", "Displacement/mm", "Force/N"]
        ))

    return runs


# Load the material and its available measurement runs.
material = "My Material"
path = FILES[material]
runs = read_instron_runs(path)

specimen = SPECIMEN_DIMENSIONS[material][0]

metadata = pd.DataFrame([
    {
        "Run": str(index + 1),
        "Width": specimen["width"],
        "Thickness": specimen["thickness"],
        "Length": specimen["gauge_length"],
    }
    for index in range(len(runs))
])

datasets = {
    material: (metadata, runs)
}

print(f"Loaded {material}: {len(runs)} run(s).")


Loaded My Material: 1 run(s).


### 2.4 Inspect the data loaded with Python

Your data has been loaded into the Notebook.
Let's take a look at the data of each material to see if it looks correct.
Inspecting small pieces of data is one of the most useful debugging and engineering-checking techniques.

- The `specimenData` table should list specimen dimensions in millimeters.
- The `runs` contain four tables that each should contain time, displacement, and force.

Run each cell below to inspect the data loaded for each of the four materials. This doesn't do anything to the data, it just lets you inspect it.

In [ ]:
# Display the specimen data and the head of run 1 for each of the four materials

for material in FILES:
    specimenData, runs = datasets[material]

    # display the specimen data for each run
    print(f"\n{material} specimen data:")
    display(specimenData)

    # display the first three rows of tensile test data for the first run
    print(f'Initial measurement data for {material} run 1:')
    display(runs[0].head(3))

## 3. Generating Plots of Stress vs Strain

The Instron tensile tester records force and displacement. In this section you will:

1. Calculate stress and strain (engineering and true) for all materials
1. Plot the calculated stresses vs strains
1. Remove the fracture tail on the engineering stress-strain curve
1. End the true stress-strain plot at the appropriate point

### 3.1 Calculate stress and strain

The first step in the data preparation is to translate from displacements and forces to the geometry-normalized quantities of (engineering) strain and stress.

Complete the calculations below in this markdown cell and in the code.

**`YOUR RESPONSE`**:

- Engineering_Strain = *your response here* 
- Engineering_Stress = *your response here*

- True_Strain = *your response here*
- True_Stress = *your response here*

***Variables for your calculation***:

- `width`
- `thickness`
- `gauge_length`
- `displacement`
- `force`

*example calculation*: wrong = (width + force * thickness)


What are the units of the calculated quantities in the equations above?

*your response here*



In [ ]:
# Retrieve the only material and run.
material = "My Material"
metadata, runs = datasets[material]

run = runs[0]
specimen = metadata.iloc[0]

width = specimen["Width"]
thickness = specimen["Thickness"]
gauge_length = specimen["Length"]

displacement = run["Displacement/mm"]
force = run["Force/N"]

# Engineering strain and stress.
run["Engineering_Strain"] = YOUR RESPONSE       # STUDENT: Replace with correct equation
run["Engineering_Stress/MPa"] = YOUR RESPONSE   # STUDENT: Replace with correct equation

# Approximate true strain and true stress.
run["True_Strain"] = YOUR RESPONSE              # STUDENT: Replace with correct equation
run["True_Stress/MPa"] = YOUR RESPONSE          # STUDENT: Replace with correct equation

print(f"Updated {material} run 1")
display(run.head(3))


### Getting ready to Plot

**Before you continue**, check out this [simple data visualizations](https://chems.gitbook.io/notebooks/data-visualization/basics_plotting) page to learn more about generating figures in Python.

We're going to generate engineering stress-strain curves in the next section. Before plotting the data, spend a little time setting up your plotting format in the code below.

**TIPS FOR PLOTTING**

*Do:*
- include captions for every figure and table.
- label axes clearly. Include units if applicable in a large enough font to be accessible to all readers.
- include a legend for figures with more than one curve or one dataset.
- if the data is presented better in multiple small graphs, combine the small graphs into one figure.
- use regression lines to demonstrate and compare data trends.
- minimize white space in graphs.

*Don’t:*
- add distractions such as unnecessary data markers, excessive grid lines, excessive tick marks, backgrounds, and 3D effects.
- plot more than 6 data sets in one graph.
- plot data sets in one graph that overlap but cannot be easily distinguished.


**Colors Options:**

- `b` (blue)
- `g` (green)
- `r` (red)
- `c` (cyan)
- `m` (magenta)
- `y` (yellow)
- `k` (black)
- `w` (white)

Modify and run the code below to see how different plotting choices affect the output.

The values you set here will be used throughout the plotting sections. Setting these values now allows you to more easily make changes across many plots in the future.


In [ ]:
# Select plotting properties for the single material in this notebook.
COLOR = {"My Material": "b"}          # Blue curve
LINE_WIDTH = {"My Material": 2.0}     # Line thickness
LINE_STYLE = {"My Material": "-"}      # Solid line

# Create example data to demonstrate how a plot is generated.
x = np.linspace(0, 1, 100)
y = 250 * x + 20 * np.sin(8 * x)

fig, ax = plt.subplots(figsize=(8, 5))

# Plot the example curve using the selected formatting options.
ax.plot(
    x,
    y,
    color=COLOR["My Material"],
    linewidth=LINE_WIDTH["My Material"],
    linestyle=LINE_STYLE["My Material"],
    label="My Material",
)

# Add descriptive labels and formatting.
ax.set_title("Example Stress–Strain Plot")
ax.set_xlabel("Engineering Strain")
ax.set_ylabel("Engineering Stress (MPa)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### 3.2 Plot engineering stress vs strain

The Instron machine continues recording data for a short time after the sample breaks.
The tail at the end of the data is not real, and needs to be removed from the plot.

1. Run the code below once to generate a plot.
1. Estimate where the engineering strain where rapid unloading/fracture tails begin for each material. 
1. Record these estimates in the `BREAK_STRAIN_CUTOFFS` for each material by replacing `None` with a decimal number.
1. Re-run the code. See if your estimate was correct.
1. Repeat steps 2–4 as needed.
1. Set the axes and title labels appropriately.



In [ ]:
# STUDENT DECISION: replace every None with the final engineering strain value that
# should remain on that material's curve. Iteratively adjust values using the graph.
BREAK_STRAIN_CUTOFF = 0.1  # STUDENT: Adjust after inspecting the plot

material = "My Material"
metadata, runs = datasets[material]
run = runs[0]

valid = run["Engineering_Strain"] <= BREAK_STRAIN_CUTOFF

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    run.loc[valid, "Engineering_Strain"],
    run.loc[valid, "Engineering_Stress/MPa"],
    color=COLOR[material],
    linewidth=LINE_WIDTH[material],
    linestyle=LINE_STYLE[material],
    label=material,
)

ax.plot(
    run.loc[~valid, "Engineering_Strain"],
    run.loc[~valid, "Engineering_Stress/MPa"],
    color=COLOR[material],
    linewidth=LINE_WIDTH[material],
    linestyle=LINE_STYLE[material],
    alpha=0.2,
    label=f"{material} (excluded)",
)

ax.set_title("Engineering Stress–Strain Curve")
ax.set_xlabel("Engineering Strain")
ax.set_ylabel("Engineering Stress (MPa)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend()
plt.show()


**`ENGINEERING CHECK`**: Inspect the plots to see where the materials began to neck. Do all the materials neck the same amount before fracture?



### 3.3 Plot True Stress and True Strain

Use the engineering stress-strain curve to determine the limit of the true stress and strain calculation.

1. Inspect the engineering stress-strain plot above
1. Estimate the engineering strain where the assumptions for the true stress and strain calculations become invalid
1. Record these estimates in the `TRUE_STRAIN_CUTOFFS` for each material by replacing `None` with the corresponding engineering strain
1. Run the code below
1. Repeat steps 2-4 as desired
1. Set the axes and title labels appropriately


In [ ]:
# STUDENT DECISION: replace with the final engineering strain cutoff.
TRUE_STRAIN_CUTOFF = 0.1       # STUDENT: adjust after inspecting the plot

material = "My Material"
metadata, runs = datasets[material]
run = runs[0]

valid = run["Engineering_Strain"] <= TRUE_STRAIN_CUTOFF

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    run.loc[valid, "True_Strain"],
    run.loc[valid, "True_Stress/MPa"],
    color=COLOR[material],
    linewidth=LINE_WIDTH[material],
    linestyle=LINE_STYLE[material],
    label=material,
)

if np.any(~valid):
    ax.plot(
        run.loc[~valid, "True_Strain"],
        run.loc[~valid, "True_Stress/MPa"],
        color=COLOR[material],
        linewidth=LINE_WIDTH[material],
        linestyle=LINE_STYLE[material],
        alpha=0.2,
        label=f"{material} (excluded)",
    )

ax.set_title("True Stress–Strain Curve", fontsize=TITLE_LABEL_FONT_SIZE)
ax.set_xlabel("True Strain", fontsize=AXIS_LABEL_FONT_SIZE)
ax.set_ylabel("True Stress (MPa)", fontsize=AXIS_LABEL_FONT_SIZE)
ax.tick_params(labelsize=TICK_LABEL_FONT_SIZE)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.show()
